# Conditional DDPM Results Analysis Notebook

This notebook is **analysis-only**.

It does not train models and does not rerun SimPEG or DPS.

It:

1. Loads `04_conditional_results.zip`.
2. Extracts all saved ID/OOD case files.
3. Computes RMSE, normalized RMSE, MAE, correlation, SSIM, gradient metrics, polarity accuracy, IoU, centroid error, area error, orientation error, and layer-interface metrics.
4. Compares SimPEG L2, unconditional DPS, oracle conditional DPS, and the equal-weight conditional mixture.
5. Generates publication-ready summary tables, scatter plots, win-rate figures, structural metric plots, uncertainty analyses, and case galleries.
6. Creates manual failure-taxonomy review sheets.
7. Exports one final ZIP containing all generated figures and CSV files.

The repository-local artifact is `analysis/artifacts/04_conditional_results.zip`. Run from the repository root with **Run All**. Kaggle attachment remains supported as a fallback.


In [ ]:
# Cell 1 — Install/import packages and configure paths
import os, sys, glob, json, zipfile, shutil, subprocess, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt

try:
    from skimage.metrics import structural_similarity
    from skimage.measure import label, regionprops
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","scikit-image"])
    from skimage.metrics import structural_similarity
    from skimage.measure import label, regionprops

from scipy.ndimage import binary_fill_holes
from scipy.stats import wilcoxon, ttest_rel, spearmanr

warnings.filterwarnings("ignore")

ZIP_NAME="04_conditional_results.zip"
REPO_ROOT=os.path.abspath(os.path.join(os.getcwd(),"..",".."))
EXTRACT_DIR=os.path.join(REPO_ROOT,"analysis","work","restored_conditional_results")
OUT_DIR=os.path.join(REPO_ROOT,"analysis","generated")
FIG_DIR=f"{OUT_DIR}/figures"
TABLE_DIR=f"{OUT_DIR}/tables"
REVIEW_DIR=f"{OUT_DIR}/failure_taxonomy_review"
FINAL_ZIP=os.path.join(REPO_ROOT,"analysis","generated","conditional_publication_analysis.zip")

for path in [EXTRACT_DIR,OUT_DIR,FIG_DIR,TABLE_DIR,REVIEW_DIR]:
    os.makedirs(path,exist_ok=True)

GEOLOGY_ORDER=["blobs","horizontal_layers","dipping_layers","faulted_layers","dykes","intrusions"]
METHOD_ARRAYS={
    "SimPEG L2":"l2",
    "Unconditional":"unconditional_mean",
    "Oracle conditional":"oracle_mean",
    "Equal mixture":"mixture_mean"
}
METHOD_PREFIX={
    "SimPEG L2":"l2",
    "Unconditional":"unconditional",
    "Oracle conditional":"oracle",
    "Equal mixture":"mixture"
}
ANOMALY_FAMILIES={"blobs","dykes","intrusions"}
LAYER_FAMILIES={"horizontal_layers","dipping_layers","faulted_layers"}
MIN_ABSOLUTE_THRESHOLD=0.20
RELATIVE_THRESHOLD=0.35

print("Analysis output:",OUT_DIR)

In [ ]:
# Cell 2 — Locate and extract 04_conditional_results.zip
def find_zip(filename):
    patterns=[
        os.path.join(REPO_ROOT,"analysis","artifacts",filename),
        f"/kaggle/working/{filename}",
        f"/kaggle/input/**/{filename}"
    ]
    matches=[]
    for pattern in patterns:
        matches.extend(glob.glob(pattern,recursive=True))
    return matches[0] if matches else None

ZIP_PATH=find_zip(ZIP_NAME)
if ZIP_PATH is None:
    raise FileNotFoundError(
        f"Attach {ZIP_NAME} to the Kaggle notebook, then rerun."
    )

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR,exist_ok=True)

with zipfile.ZipFile(ZIP_PATH,"r") as archive:
    archive.extractall(EXTRACT_DIR)

case_candidates=glob.glob(f"{EXTRACT_DIR}/**/cases",recursive=True)
case_candidates=[
    path for path in case_candidates
    if glob.glob(f"{path}/id_*.npz") and glob.glob(f"{path}/ood_*.npz")
]

if not case_candidates:
    raise FileNotFoundError(
        "The ZIP was extracted, but no cases/id_*.npz and cases/ood_*.npz files were found."
    )

CASE_DIR=max(
    case_candidates,
    key=lambda path:len(glob.glob(f"{path}/id_*.npz"))+len(glob.glob(f"{path}/ood_*.npz"))
)

RESULTS_ROOT=os.path.dirname(CASE_DIR)

print("ZIP:",ZIP_PATH)
print("CASE_DIR:",CASE_DIR)
print("ID cases:",len(glob.glob(f"{CASE_DIR}/id_*.npz")))
print("OOD cases:",len(glob.glob(f"{CASE_DIR}/ood_*.npz")))

In [ ]:
# Cell 3 — Load saved cases
def load_cases(split):
    files=sorted(glob.glob(f"{CASE_DIR}/{split}_*.npz"))
    if not files:
        raise FileNotFoundError(f"No {split.upper()} case files found.")

    records=[]
    arrays={}

    for path in files:
        with np.load(path,allow_pickle=True) as case:
            metrics=case["metrics"].item()
            case_id=int(metrics["case"])
            records.append(metrics)
            arrays[case_id]={
                key:case[key].copy()
                for key in case.files
                if key!="metrics"
            }

    df=pd.DataFrame(records).sort_values("case").reset_index(drop=True)
    df["split"]=split.upper()
    return df,arrays

id_df,id_arrays=load_cases("id")
ood_df,ood_arrays=load_cases("ood")
base_metrics=pd.concat([id_df,ood_df],ignore_index=True)

print("Loaded:",len(id_df),"ID and",len(ood_df),"OOD cases")
display(base_metrics.head())

In [ ]:
# Cell 4 — Metric functions
def safe_corr(a,b):
    a=np.asarray(a,float).ravel()
    b=np.asarray(b,float).ravel()
    if np.std(a)<1e-10 or np.std(b)<1e-10:
        return np.nan
    return float(np.corrcoef(a,b)[0,1])

def rmse(a,b):
    return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))

def normalized_rmse(truth,estimate):
    denom=float(np.sqrt(np.mean(np.asarray(truth)**2)))
    return np.nan if denom<1e-10 else rmse(truth,estimate)/denom

def compute_ssim(truth,estimate):
    truth=np.asarray(truth,float)
    estimate=np.asarray(estimate,float)
    # All models are evaluated on the fixed normalized density interval [-1, 1].
    # A common data range is required for valid cross-method SSIM comparisons.
    data_range=2.0
    return float(structural_similarity(
        truth,estimate,data_range=data_range,
        gaussian_weights=True,sigma=1.5,use_sample_covariance=False
    ))

def gradient_metrics(truth,estimate):
    tz,tx=np.gradient(np.asarray(truth,float))
    ez,ex=np.gradient(np.asarray(estimate,float))
    tm=np.sqrt(tx**2+tz**2)
    em=np.sqrt(ex**2+ez**2)
    truth_tv=float(np.mean(tm))
    est_tv=float(np.mean(em))
    return {
        "gradient_rmse":float(np.sqrt(np.mean((ex-tx)**2+(ez-tz)**2))),
        "gradient_magnitude_rmse":rmse(tm,em),
        "gradient_correlation":safe_corr(tm,em),
        "boundary_sharpness_ratio":est_tv/truth_tv if truth_tv>1e-10 else np.nan
    }

def polarity_accuracy(truth,estimate):
    threshold=max(MIN_ABSOLUTE_THRESHOLD,RELATIVE_THRESHOLD*float(np.max(np.abs(truth))))
    active=np.abs(truth)>=threshold
    return np.nan if not np.any(active) else float(np.mean(np.sign(truth[active])==np.sign(estimate[active])))

def anomaly_mask(model,threshold):
    return binary_fill_holes(np.abs(np.asarray(model))>=threshold)

def largest_region(mask):
    labeled=label(mask)
    if labeled.max()==0:
        return None
    return max(regionprops(labeled),key=lambda region:region.area)

def geometric_metrics(truth,estimate):
    threshold=max(MIN_ABSOLUTE_THRESHOLD,RELATIVE_THRESHOLD*float(np.max(np.abs(truth))))
    truth_mask=anomaly_mask(truth,threshold)
    estimate_mask=anomaly_mask(estimate,threshold)

    intersection=np.logical_and(truth_mask,estimate_mask).sum()
    union=np.logical_or(truth_mask,estimate_mask).sum()
    iou=float(intersection/union) if union>0 else np.nan

    truth_area=int(truth_mask.sum())
    estimate_area=int(estimate_mask.sum())

    truth_region=largest_region(truth_mask)
    estimate_region=largest_region(estimate_mask)

    if truth_region is not None and estimate_region is not None:
        centroid_distance=float(np.linalg.norm(
            np.asarray(truth_region.centroid)-np.asarray(estimate_region.centroid)
        ))
        centroid_distance/=np.sqrt(truth.shape[0]**2+truth.shape[1]**2)
        orientation_error=abs(np.degrees(truth_region.orientation-estimate_region.orientation))
        orientation_error=min(orientation_error,180-orientation_error)
    else:
        centroid_distance=np.nan
        orientation_error=np.nan

    return {
        "iou":iou,
        "area_ratio":estimate_area/truth_area if truth_area>0 else np.nan,
        "relative_area_error":abs(estimate_area-truth_area)/truth_area if truth_area>0 else np.nan,
        "normalized_centroid_distance":centroid_distance,
        "orientation_error_degrees":orientation_error,
        "truth_object_count":int(label(truth_mask).max()),
        "estimate_object_count":int(label(estimate_mask).max()),
        "object_count_error":abs(int(label(estimate_mask).max())-int(label(truth_mask).max()))
    }

def dominant_interfaces(model,n_interfaces=3):
    grad=np.abs(np.diff(np.asarray(model,float),axis=0))
    n_interfaces=min(n_interfaces,grad.shape[0])
    indices=np.argpartition(grad,-n_interfaces,axis=0)[-n_interfaces:]
    return np.sort(indices.astype(float)+0.5,axis=0)

def multi_interface_metrics(truth,estimate,n_interfaces=3):
    t=dominant_interfaces(truth,n_interfaces)
    e=dominant_interfaces(estimate,n_interfaces)

    per_column=[]
    for col in range(t.shape[1]):
        tc=t[:,col]
        ec=e[:,col]
        distances=[]
        for value in tc:
            distances.append(np.min(np.abs(ec-value)))
        for value in ec:
            distances.append(np.min(np.abs(tc-value)))
        per_column.append(np.mean(distances))

    per_column=np.asarray(per_column)
    return {
        "multi_interface_mean_distance_pixels":float(np.mean(per_column)),
        "multi_interface_median_distance_pixels":float(np.median(per_column)),
        "multi_interface_max_distance_pixels":float(np.max(per_column))
    }

print("Metric functions ready.")

In [ ]:
# Cell 5 — Compute structural and geological metrics for every saved case
rows=[]

for split,df,arrays in [
    ("ID",id_df,id_arrays),
    ("OOD",ood_df,ood_arrays)
]:
    for _,record in df.iterrows():
        case_id=int(record["case"])
        geology=str(record["geology_type"])
        truth=arrays[case_id]["truth"].astype(np.float32)

        for method,array_key in METHOD_ARRAYS.items():
            estimate=arrays[case_id][array_key].astype(np.float32)

            row={
                "split":split,
                "case":case_id,
                "geology_type":geology,
                "method":method,
                "rmse":rmse(truth,estimate),
                "normalized_rmse":normalized_rmse(truth,estimate),
                "mae":float(np.mean(np.abs(estimate-truth))),
                "bias":float(np.mean(estimate-truth)),
                "correlation":safe_corr(truth,estimate),
                "ssim":compute_ssim(truth,estimate),
                "polarity_accuracy":polarity_accuracy(truth,estimate)
            }

            row.update(gradient_metrics(truth,estimate))

            if geology in ANOMALY_FAMILIES:
                row.update(geometric_metrics(truth,estimate))
            else:
                for key in [
                    "iou","area_ratio","relative_area_error",
                    "normalized_centroid_distance","orientation_error_degrees",
                    "truth_object_count","estimate_object_count","object_count_error"
                ]:
                    row[key]=np.nan

            if geology in LAYER_FAMILIES:
                row.update(multi_interface_metrics(truth,estimate,n_interfaces=3))
            else:
                row["multi_interface_mean_distance_pixels"]=np.nan
                row["multi_interface_median_distance_pixels"]=np.nan
                row["multi_interface_max_distance_pixels"]=np.nan

            if method!="SimPEG L2":
                std_key=f"{METHOD_PREFIX[method]}_std"
                if std_key in arrays[case_id]:
                    posterior_std=arrays[case_id][std_key]
                    row["mean_posterior_std"]=float(np.mean(posterior_std))
                    row["uncertainty_error_spearman"]=float(
                        spearmanr(posterior_std.ravel(),np.abs(estimate-truth).ravel(),nan_policy="omit").statistic
                    )
                else:
                    row["mean_posterior_std"]=np.nan
                    row["uncertainty_error_spearman"]=np.nan
            else:
                row["mean_posterior_std"]=np.nan
                row["uncertainty_error_spearman"]=np.nan

            rows.append(row)

metrics=pd.DataFrame(rows)
metrics.to_csv(f"{TABLE_DIR}/all_case_method_metrics.csv",index=False)

print("Metric rows:",len(metrics))
display(metrics.head())

In [ ]:
# Cell 6 — Summary tables, winners and statistical tests
summary_metrics=[
    "rmse","normalized_rmse","mae","correlation","ssim",
    "gradient_rmse","gradient_correlation","boundary_sharpness_ratio",
    "polarity_accuracy","iou","relative_area_error",
    "normalized_centroid_distance","orientation_error_degrees",
    "multi_interface_mean_distance_pixels","mean_posterior_std",
    "uncertainty_error_spearman"
]

overall_summary=(
    metrics.groupby(["split","method"])[summary_metrics]
    .agg(["mean","median","std","count"])
)
overall_summary.columns=["_".join(col) for col in overall_summary.columns.to_flat_index()]
overall_summary=overall_summary.reset_index()
overall_summary.to_csv(f"{TABLE_DIR}/overall_summary.csv",index=False)

family_summary=(
    metrics.groupby(["split","geology_type","method"])[summary_metrics]
    .agg(["mean","median","std","count"])
)
family_summary.columns=["_".join(col) for col in family_summary.columns.to_flat_index()]
family_summary=family_summary.reset_index()
family_summary.to_csv(f"{TABLE_DIR}/summary_by_geology.csv",index=False)

lower_better={
    "rmse","normalized_rmse","mae","gradient_rmse",
    "relative_area_error","normalized_centroid_distance",
    "orientation_error_degrees","multi_interface_mean_distance_pixels"
}
higher_better={"correlation","ssim","gradient_correlation","polarity_accuracy","iou"}

winner_rows=[]
for (split,case_id,geology),group in metrics.groupby(["split","case","geology_type"]):
    row={"split":split,"case":case_id,"geology_type":geology}
    for metric in sorted(lower_better|higher_better):
        valid=group[["method",metric]].dropna()
        if valid.empty:
            row[f"winner_{metric}"]=np.nan
        elif metric in lower_better:
            row[f"winner_{metric}"]=valid.loc[valid[metric].idxmin(),"method"]
        else:
            row[f"winner_{metric}"]=valid.loc[valid[metric].idxmax(),"method"]
    winner_rows.append(row)

winners=pd.DataFrame(winner_rows)
winners.to_csv(f"{TABLE_DIR}/case_winners_by_metric.csv",index=False)

test_rows=[]
for split in ["ID","OOD"]:
    split_metrics=metrics[metrics.split==split]
    pivot=split_metrics.pivot(index="case",columns="method",values="rmse")
    comparisons=[
        ("Oracle conditional","Unconditional"),
        ("Equal mixture","Unconditional"),
        ("Equal mixture","Oracle conditional"),
        ("Unconditional","SimPEG L2"),
        ("Oracle conditional","SimPEG L2"),
        ("Equal mixture","SimPEG L2")
    ]
    for a,b in comparisons:
        av=pivot[a].dropna()
        bv=pivot.loc[av.index,b]
        diff=av-bv
        test_rows.append({
            "split":split,
            "metric":"rmse",
            "comparison":f"{a} vs {b}",
            "mean_difference":float(diff.mean()),
            "wilcoxon_p":float(wilcoxon(av,bv).pvalue),
            "paired_t_p":float(ttest_rel(av,bv).pvalue)
        })

statistical_tests=pd.DataFrame(test_rows)
statistical_tests.to_csv(f"{TABLE_DIR}/paired_statistical_tests.csv",index=False)

display(overall_summary)
display(family_summary.head(20))
display(statistical_tests)

In [ ]:
# Cell 7 — Publication figures: overall metrics and geology-family comparisons
def metric_boxplot(metric,label,filename,higher_better):
    fig,axes=plt.subplots(1,2,figsize=(14,5.5))
    methods=list(METHOD_ARRAYS.keys())

    for ax,split in zip(axes,["ID","OOD"]):
        subset=metrics[metrics.split==split]
        values=[subset[subset.method==method][metric].dropna() for method in methods]
        ax.boxplot(values,tick_labels=methods,showmeans=True)
        ax.set_title(f"{split}: {label} ({'higher' if higher_better else 'lower'} is better)")
        ax.set_ylabel(label)
        ax.tick_params(axis="x",rotation=20)
        ax.grid(axis="y",alpha=.3)

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{filename}",dpi=220,bbox_inches="tight")
    plt.show()

metric_boxplot("rmse","Pixelwise RMSE","overall_rmse.png",False)
metric_boxplot("ssim","SSIM","overall_ssim.png",True)
metric_boxplot("correlation","Spatial correlation","overall_correlation.png",True)
metric_boxplot("gradient_rmse","Gradient RMSE","overall_gradient_rmse.png",False)

def metric_by_geology(metric,label,filename,higher_better):
    fig,axes=plt.subplots(2,1,figsize=(14,11))
    methods=list(METHOD_ARRAYS.keys())
    width=.19

    for ax,split in zip(axes,["ID","OOD"]):
        subset=metrics[metrics.split==split]
        geology=[g for g in GEOLOGY_ORDER if g in subset.geology_type.unique()]
        x=np.arange(len(geology))

        for idx,method in enumerate(methods):
            means=[]
            sems=[]
            for geology_type in geology:
                values=subset[
                    (subset.method==method)&(subset.geology_type==geology_type)
                ][metric].dropna().to_numpy()
                means.append(np.mean(values) if len(values) else np.nan)
                sems.append(np.std(values,ddof=1)/np.sqrt(len(values)) if len(values)>1 else 0)
            ax.bar(x+(idx-1.5)*width,means,width,yerr=sems,capsize=2,label=method)

        ax.set_xticks(x)
        ax.set_xticklabels([g.replace("_"," ") for g in geology],rotation=25,ha="right")
        ax.set_ylabel(label)
        ax.set_title(f"{split}: {label} by geology ({'higher' if higher_better else 'lower'} is better)")
        ax.grid(axis="y",alpha=.3)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{filename}",dpi=220,bbox_inches="tight")
    plt.show()

metric_by_geology("ssim","Mean SSIM","ssim_by_geology.png",True)
metric_by_geology("iou","Mean anomaly IoU","iou_by_geology.png",True)
metric_by_geology("multi_interface_mean_distance_pixels","Multi-interface distance (pixels)","interface_distance_by_geology.png",False)
metric_by_geology("polarity_accuracy","Polarity accuracy","polarity_by_geology.png",True)

In [ ]:
# Cell 8 — Scatter plots, win frequencies and uncertainty analysis
markers={
    "blobs":"o","horizontal_layers":"s","dipping_layers":"^",
    "faulted_layers":"D","dykes":"v","intrusions":"P"
}

def pairwise_scatter(metric,a,b,filename):
    fig,axes=plt.subplots(1,2,figsize=(14,6))
    for ax,split in zip(axes,["ID","OOD"]):
        pivot=metrics[metrics.split==split].pivot_table(
            index=["case","geology_type"],columns="method",values=metric
        ).reset_index()

        for geology in GEOLOGY_ORDER:
            subset=pivot[pivot.geology_type==geology]
            if not subset.empty:
                ax.scatter(subset[a],subset[b],marker=markers[geology],label=geology.replace("_"," "),alpha=.75,s=45)

        values=np.concatenate([pivot[a].dropna(),pivot[b].dropna()])
        lo=float(np.min(values)); hi=float(np.max(values)); pad=.04*(hi-lo)
        ax.plot([lo-pad,hi+pad],[lo-pad,hi+pad],"--",linewidth=1.2)
        ax.set_xlim(lo-pad,hi+pad); ax.set_ylim(lo-pad,hi+pad)
        ax.set_xlabel(a); ax.set_ylabel(b)
        ax.set_title(f"{split}: {b} versus {a}")
        ax.grid(alpha=.3)
    axes[-1].legend(fontsize=8,bbox_to_anchor=(1.03,1))
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{filename}",dpi=220,bbox_inches="tight")
    plt.show()

pairwise_scatter("rmse","Unconditional","Oracle conditional","rmse_oracle_vs_unconditional.png")
pairwise_scatter("rmse","Oracle conditional","Equal mixture","rmse_mixture_vs_oracle.png")
pairwise_scatter("ssim","Unconditional","Oracle conditional","ssim_oracle_vs_unconditional.png")
pairwise_scatter("ssim","Oracle conditional","Equal mixture","ssim_mixture_vs_oracle.png")

winner_metrics={
    "RMSE":"winner_rmse",
    "SSIM":"winner_ssim",
    "Correlation":"winner_correlation",
    "IoU":"winner_iou"
}
fig,axes=plt.subplots(2,2,figsize=(15,11))
for row,split in enumerate(["ID","OOD"]):
    subset=winners[winners.split==split]
    for col,metric_group in enumerate([["RMSE","SSIM"],["Correlation","IoU"]]):
        ax=axes[row,col]
        methods=list(METHOD_ARRAYS.keys())
        x=np.arange(len(methods)); width=.36
        for idx,metric_name in enumerate(metric_group):
            percentages=[
                100*np.mean(subset[winner_metrics[metric_name]]==method)
                for method in methods
            ]
            ax.bar(x+(idx-.5)*width,percentages,width,label=metric_name)
        ax.set_xticks(x)
        ax.set_xticklabels(methods,rotation=20,ha="right")
        ax.set_ylim(0,100)
        ax.set_ylabel("Cases won (%)")
        ax.set_title(f"{split}: winner frequency")
        ax.grid(axis="y",alpha=.3)
        ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/winner_frequency.png",dpi=220,bbox_inches="tight")
plt.show()

fig,axes=plt.subplots(2,3,figsize=(17,10))
for row,split in enumerate(["ID","OOD"]):
    subset=metrics[(metrics.split==split)&(metrics.method!="SimPEG L2")]
    for col,method in enumerate(["Unconditional","Oracle conditional","Equal mixture"]):
        method_df=subset[subset.method==method]
        axes[row,col].scatter(method_df.mean_posterior_std,method_df.rmse,alpha=.7,s=40)
        valid=method_df[["mean_posterior_std","rmse"]].dropna()
        r=safe_corr(valid.mean_posterior_std,valid.rmse)
        axes[row,col].set_title(f"{split} {method}\nr={r:.3f}")
        axes[row,col].set_xlabel("Mean posterior std")
        axes[row,col].set_ylabel("RMSE")
        axes[row,col].grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/uncertainty_vs_error.png",dpi=220,bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9 — Case galleries and failure-taxonomy review sheets
def case_gallery(split,df,arrays,case_ids,title,filename):
    n=len(case_ids)
    fig,axes=plt.subplots(n,7,figsize=(22,3.4*n))
    if n==1:
        axes=np.asarray(axes)[None,:]

    for row,case_id in enumerate(case_ids):
        record=df[df.case==case_id].iloc[0]
        case=arrays[int(case_id)]
        truth=case["truth"]
        panels=[
            (truth,"Truth","RdBu_r"),
            (case["l2"],f"SimPEG\nRMSE={record.l2_model_rmse:.3f}","RdBu_r"),
            (case["unconditional_mean"],f"Unconditional\nRMSE={record.unconditional_model_rmse:.3f}","RdBu_r"),
            (case["oracle_mean"],f"Oracle\nRMSE={record.oracle_model_rmse:.3f}","RdBu_r"),
            (case["mixture_mean"],f"Mixture\nRMSE={record.mixture_model_rmse:.3f}","RdBu_r"),
            (case["oracle_std"],"Oracle std","magma"),
            (case["mixture_std"],"Mixture std","magma")
        ]
        vmax=max(np.abs(panel[0]).max() for panel in panels[:5])

        for col,(image,panel_title,cmap) in enumerate(panels):
            if cmap=="RdBu_r":
                axes[row,col].imshow(image,cmap=cmap,vmin=-vmax,vmax=vmax,origin="upper",aspect="auto")
            else:
                axes[row,col].imshow(image,cmap=cmap,vmin=0,origin="upper",aspect="auto")
            axes[row,col].set_title(panel_title,fontsize=9)
            axes[row,col].set_xticks([]); axes[row,col].set_yticks([])

        axes[row,0].set_ylabel(
            f"{split} case {case_id}\n{record.geology_type.replace('_',' ')}",
            fontsize=8
        )

    fig.suptitle(title,fontsize=15,y=1.002)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{filename}",dpi=200,bbox_inches="tight")
    plt.close(fig)

for split,df,arrays in [("ID",id_df,id_arrays),("OOD",ood_df,ood_arrays)]:
    work=df.copy()
    work["oracle_gain"]=work.unconditional_model_rmse-work.oracle_model_rmse
    work["mixture_gain"]=work.unconditional_model_rmse-work.mixture_model_rmse
    work["mixture_vs_oracle"]=work.oracle_model_rmse-work.mixture_model_rmse

    selections={
        "largest_oracle_gains":work.nlargest(6,"oracle_gain").case.astype(int).tolist(),
        "largest_oracle_losses":work.nsmallest(6,"oracle_gain").case.astype(int).tolist(),
        "mixture_beats_oracle":work.nlargest(6,"mixture_vs_oracle").case.astype(int).tolist(),
        "oracle_beats_mixture":work.nsmallest(6,"mixture_vs_oracle").case.astype(int).tolist()
    }

    for name,case_ids in selections.items():
        case_gallery(split,work,arrays,case_ids,f"{split}: {name.replace('_',' ')}",f"{split.lower()}_{name}.png")

failure_labels=[
    "wrong_family","wrong_polarity","oversmoothing","interface_displacement",
    "missed_anomaly","extra_anomaly","wrong_location","wrong_size",
    "incorrect_fault_geometry","incorrect_dip","amplitude_underestimate",
    "amplitude_overestimate","acceptable","other"
]

review_rows=[]
for split,df in [("ID",id_df),("OOD",ood_df)]:
    work=df.copy()
    work["oracle_gain"]=work.unconditional_model_rmse-work.oracle_model_rmse
    work["mixture_vs_oracle"]=work.oracle_model_rmse-work.mixture_model_rmse

    selected=pd.concat([
        work.nlargest(8,"oracle_gain"),
        work.nsmallest(8,"oracle_gain"),
        work.nlargest(8,"mixture_vs_oracle"),
        work.nsmallest(8,"mixture_vs_oracle")
    ]).drop_duplicates("case")

    for _,row in selected.iterrows():
        item={
            "split":split,
            "case":int(row.case),
            "geology_type":row.geology_type,
            "l2_rmse":row.l2_model_rmse,
            "unconditional_rmse":row.unconditional_model_rmse,
            "oracle_rmse":row.oracle_model_rmse,
            "mixture_rmse":row.mixture_model_rmse,
            "preferred_method_visual":"",
            "confidence_1_to_5":"",
            "review_notes":""
        }
        for label_name in failure_labels:
            item[f"failure_{label_name}"]=""
        review_rows.append(item)

review_template=pd.DataFrame(review_rows)
review_template.to_csv(f"{TABLE_DIR}/failure_taxonomy_template.csv",index=False)

print("Case galleries and review template created.")

In [ ]:
# Cell 10 — Export complete analysis ZIP
metadata={
    "source_zip":ZIP_PATH,
    "case_dir":CASE_DIR,
    "n_id_cases":len(id_df),
    "n_ood_cases":len(ood_df),
    "geology_order":GEOLOGY_ORDER,
    "methods":list(METHOD_ARRAYS.keys()),
    "anomaly_threshold_absolute":MIN_ABSOLUTE_THRESHOLD,
    "anomaly_threshold_relative":RELATIVE_THRESHOLD
}
with open(f"{OUT_DIR}/analysis_metadata.json","w") as f:
    json.dump(metadata,f,indent=2)

if os.path.exists(FINAL_ZIP):
    os.remove(FINAL_ZIP)

with zipfile.ZipFile(FINAL_ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for root,_,files in os.walk(OUT_DIR):
        for filename in sorted(files):
            path=os.path.join(root,filename)
            archive.write(path,arcname=os.path.relpath(path,OUT_DIR))

generated=[
    path for path in glob.glob(f"{OUT_DIR}/**/*",recursive=True)
    if os.path.isfile(path)
]

print("="*72)
print("ANALYSIS COMPLETE")
print("="*72)
print("Generated files:",len(generated))
print("Output folder:",OUT_DIR)
print("ZIP size:",f"{os.path.getsize(FINAL_ZIP)/(1024**2):.1f} MB")
print("Download:",FINAL_ZIP)